# AISE 4010 Project, Group 11: Stage 2
### Lab 6 recurrent models (SimpleRNN, LSTM, GRU) and a TCN on the Stage 1 data

Stage 2 trains the six recurrent layouts from Lab 6 and a temporal convolutional network (TCN) on **exactly the same data as Stage 1**: hourly resampling, causal imputation, features, chronological 60/20/20 split, `profile` seasonal residual, dropped columns, train-only scaler, and 168 h -> 24 h windows. The Stage 1 data cells below are copied unchanged from `AISE4010_Project_Group11.ipynb`, and the result is checked against the scaler and feature list saved in `models/stage1_best.pt`.

Every model gets the same one-stage grid search (20 trials), starting from its 5 best configurations of the first Stage 2 search (`models/stage2_lab6_trials.csv`). The winners are retrained with 3 seeds. Selection uses Stage 1's score: **validation MAE on measured hours of midnight-origin forecasts**. The test split is not used in this notebook.

Set `QUICK = True` for a fast end-to-end check (2 epochs, one trial and one seed per model).

In [ ]:
QUICK = False  # True: 2 epochs and the first trial of each search stage, for a fast end-to-end check

## Data (identical to Stage 1)
Cells copied verbatim from the Stage 1 notebook.

In [ ]:
import os

# Convert the raw UCI .txt (';'-separated, '?' = missing) to CSV once
if not os.path.exists('household_power_consumption.csv'):
    with open('household_power_consumption.txt', newline='') as src, open('household_power_consumption.csv', 'w', newline='') as dst:
        for line in src:
            dst.write(line.replace(';', ',').replace('?', ''))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('household_power_consumption.csv')
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df = df.drop(columns=['Date', 'Time']).set_index('Datetime')
df.head()

### GPU check

In [ ]:
import os
import random
import sys
import numpy as np

os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ.get('PATH', '')
import torch

print('PyTorch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA version:', torch.version.cuda, '| cuDNN version:', torch.backends.cudnn.version())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print('Training on GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU found, training on CPU')

# A small matmul proves tensors actually land on the chosen device
probe = torch.randn(256, 256, device=device)
print('Test matmul ran on:', (probe @ probe).device)

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # also seeds CUDA

# Deterministic cuDNN kernels so GPU runs repeat exactly with the same seed
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed()

In [ ]:
TARGET = 'Global_active_power'
LOOKBACK, HORIZON = 168, 24
DAY, WEEK = 24, 168  # seasonal periods in hours

### Resampling and imputation
Hourly mean (at least 30 recorded minutes), short gaps forward-filled, long gaps filled from one week earlier. All fills are causal.

In [ ]:
counts = df[TARGET].resample('h').count()
hourly_raw = df.resample('h').mean()
hourly_raw.loc[counts < 30] = np.nan
imputed = hourly_raw[TARGET].isna()

# Length of each run of missing hours
run_id = (imputed != imputed.shift()).cumsum()
run_len = imputed.groupby(run_id).transform('size')
short_gap = imputed & (run_len <= 3)

hourly = hourly_raw.copy()
hourly.loc[short_gap] = hourly_raw.ffill().loc[short_gap]
for _ in range(2):  # a second pass covers gaps whose previous week was also missing
    hourly = hourly.fillna(hourly.shift(WEEK))
hourly = hourly.ffill().dropna()  # only leading hours with no history can remain; drop them
imputed = imputed.loc[hourly.index]

assert not hourly.isna().any().any()
print(f'Hourly rows: {len(hourly)} | imputed hours: {imputed.sum()} ({imputed.mean():.2%}), '
      f'of which short gaps: {short_gap.sum()}')

### Features and chronological split

In [ ]:
features = hourly.copy()
idx = features.index
dow = idx.day_of_week.to_numpy()
year_phase = 2 * np.pi * ((idx.day_of_year.to_numpy() - 1) + idx.hour.to_numpy() / 24) / 365.2425
features['hour_sin'] = np.sin(2 * np.pi * idx.hour / 24)
features['hour_cos'] = np.cos(2 * np.pi * idx.hour / 24)
features['dow_sin'] = np.sin(2 * np.pi * dow / 7)
features['dow_cos'] = np.cos(2 * np.pi * dow / 7)
features['is_weekend'] = (dow >= 5).astype(float)
for harmonic in (1, 2):
    features[f'year_sin_{harmonic}'] = np.sin(harmonic * year_phase)
    features[f'year_cos_{harmonic}'] = np.cos(harmonic * year_phase)

past_gap = hourly[TARGET]
causal_mean = past_gap.shift(1).expanding(min_periods=1).mean().fillna(past_gap.iloc[0])
annual_index = idx - pd.DateOffset(years=1)
annual_lag = past_gap.reindex(annual_index).to_numpy()
features['GAP_lag_1y_available'] = np.isfinite(annual_lag).astype(float)
features['GAP_lag_1y'] = np.where(np.isfinite(annual_lag), annual_lag, causal_mean.to_numpy())
for days in (7, 28):
    trailing = past_gap.shift(1).rolling(days * 24, min_periods=24).mean()
    features[f'GAP_mean_{days}d'] = trailing.fillna(causal_mean)
FEATURES = list(features.columns)
TARGET_IDX = FEATURES.index(TARGET)

n = len(features)
train_end, val_end = int(n * 0.6), int(n * 0.8)
splits = {'train': (0, train_end), 'val': (train_end, val_end), 'test': (val_end, n)}

pd.DataFrame([{'split': s, 'start': features.index[a], 'end': features.index[b - 1], 'hours': b - a,
               'imputed %': round(imputed.iloc[a:b].mean() * 100, 2)}
              for s, (a, b) in splits.items()]).set_index('split')

### Causal seasonal baselines and windows

In [ ]:
gap = hourly[TARGET].to_numpy()
how = (features.index.day_of_week * 24 + features.index.hour).to_numpy()  # hour of the week, 0-167
profile = pd.Series(gap[:train_end]).groupby(how[:train_end]).mean()  # train rows only
weekly_profile = profile.reindex(how).to_numpy()
train_residual = gap[:train_end] - weekly_profile[:train_end]
month_effect = pd.Series(train_residual).groupby(features.index.month[:train_end]).mean()
annual_adjustment = pd.Series(features.index.month).map(month_effect).fillna(0.0).to_numpy()
annual_profile = weekly_profile + annual_adjustment
seasonal_base = {
    'none': np.zeros(n),
    'week': np.r_[np.full(WEEK, np.nan), gap[:-WEEK]],  # same hour one week earlier
    'profile': weekly_profile,
    'profile_year': annual_profile,
}

In [ ]:
from sklearn.preprocessing import StandardScaler

def to_kw(y, split):
    """Undo the target scaling and add the seasonal baseline back, so all metrics are in kW."""
    return y * x_scaler.scale_[TARGET_IDX] + x_scaler.mean_[TARGET_IDX] + base[split]

def make_windows(start, end):
    """Windows whose 24-h target lies in rows [start, end); the lookback may come from earlier rows."""
    # The 'week' residual needs one extra week of history; every variant uses the same windows so they compare fairly
    t = np.arange(max(start, LOOKBACK + WEEK), end - HORIZON + 1)
    X = np.stack([X_all[i - LOOKBACK:i] for i in t])
    y = np.stack([X_all[i:i + HORIZON, TARGET_IDX] for i in t])
    return X, y, t

def prepare(variant, drop):
    """Scale the inputs and build the windows for one seasonal baseline and column set (sets the globals used below)."""
    global FEATURES, TARGET_IDX, x_scaler, X_all, data, base, measured, daily_origin, Y_kw, X_train, y_train, X_val, y_val
    frame = features.drop(columns=drop)
    target_col = TARGET
    if variant != 'none':
        frame['GAP_resid'] = gap - seasonal_base[variant]  # raw GAP stays as a lagged input
        target_col = 'GAP_resid'
    FEATURES = list(frame.columns)
    TARGET_IDX = FEATURES.index(target_col)
    x_scaler = StandardScaler().fit(frame.iloc[:train_end])  # train rows only; NaNs (first week of 'week') are ignored
    X_all = x_scaler.transform(frame).astype(np.float32)

    data = {s: make_windows(a, b) for s, (a, b) in splits.items()}
    hours = {s: t[:, None] + np.arange(HORIZON) for s, (_, _, t) in data.items()}  # row index of every target hour
    base = {s: seasonal_base[variant][h] for s, h in hours.items()}
    measured = {s: ~imputed.to_numpy()[h] for s, h in hours.items()}
    daily_origin = {s: features.index[t].hour.to_numpy() == 0 for s, (_, _, t) in data.items()}
    Y_kw = {s: to_kw(y, s) for s, (_, y, _) in data.items()}

    X_train, y_train, t_train = data['train']
    X_val, y_val, t_val = data['val']
    assert X_train.shape[1:] == (LOOKBACK, len(FEATURES)) and y_train.shape[1] == HORIZON
    assert t_train[-1] + HORIZON <= train_end and t_val[-1] + HORIZON <= val_end  # no target crosses a split
    assert all(np.isfinite(X).all() for X, _, _ in data.values())
    for s, h in hours.items():  # adding the baseline back recovers the actual kW
        assert np.allclose(Y_kw[s], gap[h], atol=1e-3)

### Use the Stage 1 configuration and check the data is identical
The Stage 1 ablation selected the seasonal baseline and the dropped columns; both are read from the saved Stage 1 checkpoint rather than re-run. The asserts confirm the feature list and the train-only scaler match Stage 1 exactly, so every model below sees the same windows.

In [ ]:
ckpt = torch.load('models/stage1_best.pt', map_location=device)
prepare(ckpt['variant'], ckpt['dropped'])

assert (LOOKBACK, HORIZON, TARGET) == (ckpt['lookback'], ckpt['horizon'], ckpt['target'])
assert FEATURES == ckpt['features'], 'feature list differs from Stage 1'
assert np.allclose(x_scaler.mean_, ckpt['x_mean']) and np.allclose(x_scaler.scale_, ckpt['x_scale']), 'scaler differs from Stage 1'

print(f"Seasonal baseline {ckpt['variant']!r}, dropped {ckpt['dropped']} -> {len(FEATURES)} features")
for s, (X, y, t) in data.items():
    print(f'{s:5s}  X {X.shape}  y {y.shape}')
print('Data identical to Stage 1: features, scaler and windows match the Stage 1 checkpoint.')

## Reference models (Stage 1)
Seasonal naive (day and week), Ridge with the Stage 1 alpha (1e4), and the tuned Stage 1 network reloaded from its checkpoint. All scores are validation only.

In [ ]:
import copy
import itertools
import os
import time
from sklearn.linear_model import Ridge
from torch import nn

def evaluate(y_true, y_pred, mask=None):
    err = y_pred - y_true
    if mask is not None:
        err = err[mask]
    return {'MAE': np.abs(err).mean(), 'RMSE': np.sqrt((err ** 2).mean())}

def flat(X):
    return X.reshape(len(X), -1)

DAILY = measured['val'] & daily_origin['val'][:, None]  # primary score: midnight origins, measured hours

def val_score(pred_kw):
    daily = evaluate(Y_kw['val'], pred_kw, DAILY)
    return {'val MAE': daily['MAE'], 'val RMSE': daily['RMSE'],
            'val MAE (all origins)': evaluate(Y_kw['val'], pred_kw, measured['val'])['MAE']}

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(xb.to(device)).cpu() for xb in torch.from_numpy(X).split(1024)]).numpy()

In [ ]:
# Stage 1 network classes, copied verbatim so the Stage 1 checkpoint can be reloaded
class MLP(nn.Module):
    def __init__(self, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        dims = [LOOKBACK * len(FEATURES)] + [hidden] * layers
        blocks = [nn.Flatten()]
        for d_in, d_out in zip(dims, dims[1:]):
            blocks += [nn.Linear(d_in, d_out), nn.BatchNorm1d(d_out), nn.ReLU(), nn.Dropout(dropout)]
        self.net = nn.Sequential(*blocks, nn.Linear(dims[-1], HORIZON))

    def forward(self, x):
        return self.net(x)

class CNN(nn.Module):
    def __init__(self, filters=64, kernel=5, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(len(FEATURES), filters, kernel, padding='same'), nn.BatchNorm1d(filters), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(filters, filters, kernel, padding='same'), nn.BatchNorm1d(filters), nn.ReLU(), nn.MaxPool1d(2),
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(filters * (LOOKBACK // 4), HORIZON))

    def forward(self, x):
        return self.net(x.transpose(1, 2))  # (batch, time, features) -> (batch, channels, time)

In [ ]:
t_val = data['val'][2]
val_pred = {  # model name -> validation forecasts in kW
    'Seasonal naive (day)': gap[t_val[:, None] - DAY + np.arange(HORIZON)],
    'Seasonal naive (week)': gap[t_val[:, None] - WEEK + np.arange(HORIZON)],
}
predict_fn = {}  # model name -> function(X windows) -> scaled forecasts, used for inference timing

ridge = Ridge(alpha=1e4).fit(flat(X_train), y_train)
val_pred['Ridge'] = to_kw(ridge.predict(flat(X_val)), 'val')
predict_fn['Ridge'] = lambda X: ridge.predict(flat(X))

stage1_name = f"Stage 1 {ckpt['family']} (tuned)"
stage1_model = {'MLP': MLP, 'CNN': CNN}[ckpt['family']](**ckpt['config']).to(device)
stage1_model.load_state_dict(ckpt['state_dict'])
val_pred[stage1_name] = to_kw(predict(stage1_model, X_val), 'val')
predict_fn[stage1_name] = lambda X: predict(stage1_model, X)

reference = pd.DataFrame({name: val_score(p) for name, p in val_pred.items()}).T
reference['parameters'] = [0, 0, ridge.coef_.size + ridge.intercept_.size, count_params(stage1_model)]
REFERENCES = list(reference.index)
reference.round(4)

## Models
PyTorch versions of the Lab 6 Keras models. Lab 6 forecast one hour with `Dense(1)`; here the last layer outputs all 24 forecast hours.

| Model | Layout (hidden = h) | Depth searched |
|---|---|---|
| Vanilla LSTM | `LSTM(h)` | 1, 2 |
| Stacked LSTM | `LSTM(h)` -> `LSTM(h/2)` (-> `LSTM(h/2)`) | 2, 3 |
| Bidirectional LSTM | `Bidirectional(LSTM(h/2))` per layer | 1, 2 |
| LSTM + Dropout + Dense | `LSTM(h)`, `Dropout`, `Dense(h/2, relu)` | 1, 2 |
| Stacked GRU | `GRU(h)` -> `GRU(h/2)` (-> `GRU(h/2)`) | 2, 3 |
| Stacked SimpleRNN | `RNN(h)` -> `RNN(h/2)` (-> `RNN(h/2)`) | 2, 3 |
| TCN | causal dilated residual blocks, kernel 7, h channels | 4, 6 levels |

Depth adds recurrent layers to each Lab 6 layout: later layers use h/2 units, as in Lab 6's 64 -> 32. Dropout is applied between layers and before the output. The **bidirectional** model reads the 168-hour input window in both directions. That window lies entirely in the past, so it sees no future values. As in Keras, the output joins the forward state at the last hour with the backward state at the first hour. The **TCN** uses causal convolutions (each output depends only on earlier hours). Dilations 1, 2, 4, ... give a receptive field of 181 h with 4 levels and 757 h with 6, so both cover the one-week lookback.

In [ ]:
class Recurrent(nn.Module):
    """Lab 6 recurrent layouts: `depth` layers (hidden, then hidden // 2 units), last time step -> 24-hour output."""
    def __init__(self, cell, hidden, depth, dropout, bidirectional=False, dense_head=False):
        super().__init__()
        rnn = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[cell]
        dirs = 2 if bidirectional else 1
        sizes = [hidden // dirs] + [hidden // (2 * dirs)] * (depth - 1)  # units per direction
        inputs = [len(FEATURES)] + [s * dirs for s in sizes[:-1]]
        self.layers = nn.ModuleList(rnn(i, s, batch_first=True, bidirectional=bidirectional) for i, s in zip(inputs, sizes))
        self.bidirectional, self.last = bidirectional, sizes[-1]
        self.drop = nn.Dropout(dropout)
        out = sizes[-1] * dirs
        self.head = (nn.Sequential(nn.Linear(out, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, HORIZON))
                     if dense_head else nn.Linear(out, HORIZON))

    def forward(self, x):
        for layer in self.layers:
            x = self.drop(layer(x)[0])
        if self.bidirectional:  # Keras-style: forward state after the last hour, backward state after the first
            return self.head(torch.cat([x[:, -1, :self.last], x[:, 0, self.last:]], dim=-1))
        return self.head(x[:, -1])

class CausalConv1d(nn.Module):
    def __init__(self, c_in, c_out, kernel, dilation):
        super().__init__()
        self.pad = (kernel - 1) * dilation
        self.conv = nn.Conv1d(c_in, c_out, kernel, padding=self.pad, dilation=dilation)

    def forward(self, x):
        return self.conv(x)[..., :-self.pad]  # drop the right padding so no output sees a later hour

class TemporalBlock(nn.Module):
    def __init__(self, c_in, c_out, kernel, dilation, dropout):
        super().__init__()
        self.net = nn.Sequential(CausalConv1d(c_in, c_out, kernel, dilation), nn.ReLU(), nn.Dropout(dropout),
                                 CausalConv1d(c_out, c_out, kernel, dilation), nn.ReLU(), nn.Dropout(dropout))
        self.skip = nn.Conv1d(c_in, c_out, 1) if c_in != c_out else nn.Identity()

    def forward(self, x):
        return torch.relu(self.net(x) + self.skip(x))

TCN_KERNEL = 7

class TCN(nn.Module):
    def __init__(self, hidden, depth, dropout, kernel=TCN_KERNEL):
        super().__init__()
        assert 1 + 2 * (kernel - 1) * (2 ** depth - 1) >= LOOKBACK, 'receptive field must cover the lookback'
        channels = [len(FEATURES)] + [hidden] * depth
        self.net = nn.Sequential(*[TemporalBlock(a, b, kernel, 2 ** i, dropout)
                                   for i, (a, b) in enumerate(zip(channels, channels[1:]))])
        self.head = nn.Linear(hidden, HORIZON)

    def forward(self, x):
        return self.head(self.net(x.transpose(1, 2))[:, :, -1])  # (batch, time, features) -> channels first

# name -> (builder taking hidden/depth/dropout, depth values searched)
FAMILIES = {
    'Vanilla LSTM':           (lambda **c: Recurrent('LSTM', **c), (1, 2)),
    'Stacked LSTM':           (lambda **c: Recurrent('LSTM', **c), (2, 3)),
    'Bidirectional LSTM':     (lambda **c: Recurrent('LSTM', bidirectional=True, **c), (1, 2)),
    'LSTM + Dropout + Dense': (lambda **c: Recurrent('LSTM', dense_head=True, **c), (1, 2)),
    'Stacked GRU':            (lambda **c: Recurrent('GRU', **c), (2, 3)),
    'Stacked SimpleRNN':      (lambda **c: Recurrent('RNN', **c), (2, 3)),
    'TCN':                    (lambda **c: TCN(**c), (4, 6)),
}

# Shape and causality checks: every model maps (batch, 168, F) -> (batch, 24), and the TCN ignores later hours
probe = torch.from_numpy(X_val[:4]).to(device)
for name, (build, depths) in FAMILIES.items():
    for depth in depths:
        m = build(hidden=32, depth=depth, dropout=0.0).to(device).eval()
        assert m(probe).shape == (4, HORIZON), name
tcn_probe = TCN(hidden=8, depth=4, dropout=0.0).to(device).eval()
with torch.no_grad():
    changed = probe.clone(); changed[:, -1] += 1.0
    full_a, full_b = tcn_probe.net(probe.transpose(1, 2)), tcn_probe.net(changed.transpose(1, 2))
assert torch.allclose(full_a[..., :-1], full_b[..., :-1]), 'TCN output depends on a later hour'
print('All 7 models produce (batch, 24) forecasts; the TCN is causal.')

## Training
Same procedure for every model: **Adam** (no weight decay) with gradient clipping at 1.0 (Lab 6's `clipnorm`), and a **step learning-rate schedule** that multiplies the rate by 0.75 (gamma) every `step` epochs, never going below 1e-6. The loss on the scaled target is a search option (L1 or Huber). Early stopping uses the same loss on the validation windows. Each run has at most 100 epochs and stops after **15 epochs** without validation improvement, and the weights from the best validation epoch are restored.

In [ ]:
MAX_EPOCHS, PATIENCE = (2, 2) if QUICK else (100, 15)
LR_GAMMA = 0.75
MIN_LR = 1e-6
LOSSES = {'MSE': nn.MSELoss, 'L1': nn.L1Loss, 'Huber': nn.HuberLoss}

def fit(build, train, val, lr, batch_size, step, loss='MSE', seed=SEED, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """Train with early stopping on the validation loss; returns the best model, loss history and train time."""
    set_seed(seed)
    model = build().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    # Step decay (same as StepLR) with a floor at MIN_LR
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: max(LR_GAMMA ** (e // step), MIN_LR / lr))
    loss_fn = LOSSES[loss]()
    X_tr, y_tr = (torch.from_numpy(a).to(device) for a in train)  # copy the data to the device once
    X_va, y_va = (torch.from_numpy(a).to(device) for a in val)
    history = {'train': [], 'val': []}
    best_loss, best_state, wait = float('inf'), copy.deepcopy(model.state_dict()), 0
    start = time.perf_counter()
    for _ in range(max_epochs):
        model.train()
        total = 0.0
        for idx in torch.randperm(len(X_tr), device=device).split(batch_size):
            opt.zero_grad()
            loss = loss_fn(model(X_tr[idx]), y_tr[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.detach() * len(idx)
        sched.step()
        model.eval()
        with torch.no_grad():
            val_loss = sum(loss_fn(model(xb), yb).item() * len(xb)
                           for xb, yb in zip(X_va.split(1024), y_va.split(1024))) / len(X_va)
        history['train'].append(total.item() / len(X_tr))
        history['val'].append(val_loss)
        if val_loss < best_loss:
            best_loss, wait = val_loss, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, history, time.perf_counter() - start

## One-stage grid search (validation only)
The first Stage 2 run (a 30-trial two-stage grid per model, MSE loss, learning rate halved every 5, 7 or 10 epochs) is logged in `models/stage2_lab6_trials.csv`. It showed two things:
* Nearly every run reached its best validation score by **epoch 1 to 5**, before the first learning-rate drop. The schedule therefore never changed a result: trials that differed only in `step` scored identically.
* Training minimised MSE while models are scored on MAE.

This search starts from each model's **5 best distinct configurations** from that log (hidden, depth, dropout, learning rate, batch size) and tests both fixes in a single stage:
* **Learning-rate step** {1, 2} epochs, with the rate multiplied by 0.75 at each step (gamma 0.75): the decay starts while the model is still improving.
* **Loss** {L1, Huber}: L1 trains directly on absolute error, the quantity being scored; Huber is squared for small errors and linear for large ones.

That is 5 x 2 x 2 = **20 trials per model** (140 in total), all with seed 42. Objective: validation MAE (kW) on measured hours of midnight-origin forecasts. Each trial is appended to `models/stage2_lab6_refine.csv`, and re-running the cell skips trials already in the file.

In [ ]:
ARCH, TRAIN = ['hidden', 'depth', 'dropout'], ['lr', 'batch_size', 'step', 'loss']
TOP_CONFIGS = 5
GRID = {'step': [1, 2], 'loss': ['L1', 'Huber']}

FIRST_LOG = 'models/stage2_lab6_trials.csv'
assert os.path.exists(FIRST_LOG), f'{FIRST_LOG} (the first Stage 2 search log) is needed for the starting configurations'
first = pd.read_csv(FIRST_LOG)
# `step` never changed a result in the first search, so configurations that differ only in step are duplicates
starts = (first.sort_values('val MAE').drop_duplicates(['family', 'hidden', 'depth', 'dropout', 'lr', 'batch_size'])
          .groupby('family', sort=False).head(1 if QUICK else TOP_CONFIGS))
first_best = first.groupby('family')['val MAE'].min()

LOG = 'models/stage2_lab6_refine' + ('_quick' if QUICK else '') + '.csv'
if QUICK and os.path.exists(LOG):
    os.remove(LOG)  # the quick check always starts fresh and never touches the real log

def trial_key(family, cfg):
    return '|'.join([family] + [str(cfg[k]) for k in ARCH + TRAIN])

def load_log():
    return pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame(columns=['key'])

def run_trial(family, cfg):
    if trial_key(family, cfg) in set(load_log()['key']):
        return
    build = FAMILIES[family][0]
    model, hist, secs = fit(lambda: build(**{k: cfg[k] for k in ARCH}), (X_train, y_train), (X_val, y_val),
                            **{k: cfg[k] for k in TRAIN})
    row = {'key': trial_key(family, cfg), 'family': family, **cfg,
           **val_score(to_kw(predict(model, X_val), 'val')),
           'epochs': len(hist['val']), 'best epoch': int(np.argmin(hist['val'])) + 1,
           'train time (s)': secs, 'parameters': count_params(model)}
    pd.DataFrame([row]).to_csv(LOG, mode='a', header=not os.path.exists(LOG), index=False)
    print(f"{family:24s} {cfg} -> val MAE {row['val MAE']:.4f} kW ({row['epochs']} epochs, {secs:.0f} s)")

grid = list(itertools.product(*GRID.values()))
for family in FAMILIES:
    for _, s in starts[starts['family'] == family].iterrows():
        start_cfg = {'hidden': int(s['hidden']), 'depth': int(s['depth']), 'dropout': float(s['dropout']),
                     'lr': float(s['lr']), 'batch_size': int(s['batch_size'])}  # not `base`: that is Stage 1's seasonal baseline
        for step, loss in (grid[:1] if QUICK else grid):
            run_trial(family, {**start_cfg, 'step': step, 'loss': loss})

trials = load_log()
print(f'{len(trials)} trials logged in {LOG}')

In [ ]:
# Best configuration per model, compared with the best of the first search
best_cfg = (trials.sort_values('val MAE').groupby('family', sort=False).head(1)
            .set_index('family').loc[list(FAMILIES)])
best_cfg['first search val MAE'] = first_best
best_cfg['change (kW)'] = best_cfg['val MAE'] - best_cfg['first search val MAE']
best_cfg[ARCH + TRAIN + ['val MAE', 'first search val MAE', 'change (kW)', 'best epoch', 'parameters']].sort_values('val MAE').round(4)

### Retrain the selected configurations with 3 seeds
The search log keeps scores, not weights, so each model's winning configuration is retrained with seeds 42, 1 and 2. The mean and standard deviation over the seeds show whether gaps of a few thousandths of a kW between models are larger than run-to-run noise. The seed-42 run reproduces the search trial; its weights and loss history are used for the plots below.

In [ ]:
SEEDS = [SEED] if QUICK else [SEED, 1, 2]
tuned, histories, train_secs, seed_mae = {}, {}, {}, {}
for family, cfg in best_cfg.iterrows():
    build = FAMILIES[family][0]
    arch = {'hidden': int(cfg['hidden']), 'depth': int(cfg['depth']), 'dropout': float(cfg['dropout'])}
    train_cfg = {'lr': float(cfg['lr']), 'batch_size': int(cfg['batch_size']), 'step': int(cfg['step']), 'loss': cfg['loss']}
    seed_mae[family] = []
    for seed in SEEDS:
        model, hist, secs = fit(lambda: build(**arch), (X_train, y_train), (X_val, y_val), seed=seed, **train_cfg)
        pred = to_kw(predict(model, X_val), 'val')
        seed_mae[family].append(val_score(pred)['val MAE'])
        if seed == SEED:
            tuned[family], histories[family], train_secs[family], val_pred[family] = model, hist, secs, pred
            predict_fn[family] = lambda X, m=model: predict(m, X)
    print(f"{family:24s} search {cfg['val MAE']:.4f} | seeds {np.round(seed_mae[family], 4)} -> "
          f"{np.mean(seed_mae[family]):.4f} +/- {np.std(seed_mae[family]):.4f} kW")

seed_table = pd.DataFrame({f: {'val MAE mean (3 seeds)': np.mean(v), 'val MAE std (3 seeds)': np.std(v)}
                           for f, v in seed_mae.items()}).T

## Results (validation only)
### Learning curves

In [ ]:
def plot_losses(histories, columns=3):
    rows = -(-len(histories) // columns)  # ceiling division
    fig, axes = plt.subplots(rows, columns, figsize=(4.6 * columns, 3.3 * rows), squeeze=False)
    for ax, (name, h) in zip(axes.flat, histories.items()):
        epochs = np.arange(1, len(h['train']) + 1)
        ax.plot(epochs, h['train'], label='training')
        ax.plot(epochs, h['val'], label='validation')
        ax.axvline(np.argmin(h['val']) + 1, color='grey', ls=':', label='best epoch')
        ax.set_title(name); ax.set_xlabel('epoch'); ax.set_yscale('log'); ax.grid(alpha=0.3)
    axes.flat[0].legend()
    for ax in axes[:, 0]:
        ax.set_ylabel('MSE loss (scaled, log)')
    for ax in list(axes.flat)[len(histories):]:
        ax.axis('off')
    fig.suptitle('Fig. 1 - Learning curves of the selected configurations')
    plt.tight_layout(); plt.show()

plot_losses(histories)

If the validation loss bottoms out early while the training loss keeps falling, the model is overfitting. The dotted line marks the restored best epoch.

### Grid-search trials

In [ ]:
fig, axes = plt.subplots(len(FAMILIES), 1, figsize=(14, 2.6 * len(FAMILIES)), squeeze=False)
for ax, family in zip(axes[:, 0], FAMILIES):
    rows = trials[trials['family'] == family].reset_index(drop=True)
    colors = np.where(rows['loss'] == 'L1', 'C0', 'C1').astype(object)
    colors[rows['val MAE'].idxmin()] = 'C3'
    ax.bar(range(len(rows)), rows['val MAE'], color=colors)
    ax.axhline(first_best[family], color='C2', ls='-', lw=1, label='best of first search (MSE loss)')
    ax.axhline(reference.loc[stage1_name, 'val MAE'], color='k', ls='--', lw=1, label=stage1_name)
    labels = rows.apply(lambda r: f"h{r['hidden']} d{r['depth']} p{r['dropout']} lr{r['lr']:g} b{r['batch_size']} | s{r['step']}", axis=1)
    ax.set_xticks(range(len(rows)), labels, rotation=90, fontsize=6)
    vals = np.r_[rows['val MAE'], first_best[family], reference.loc[stage1_name, 'val MAE']]
    lo, hi = vals.min(), vals.max()
    ax.set_ylim(lo - 0.05 * (hi - lo + 1e-3), hi + 0.05 * (hi - lo + 1e-3))
    ax.set(ylabel='val MAE (kW)', title=f'{family}: L1 loss (blue), Huber loss (orange), best (red)')
axes[0, 0].legend(fontsize=8)
fig.suptitle('Fig. 2 - Grid-search trials per model (h hidden, d depth, p dropout, b batch, s lr step)')
plt.tight_layout(); plt.show()

### Inference time
Two measurements on the same device for every model: the cost of a batch of 1,024 windows (per 1,000 windows), and the median latency of one-window forecasts (200 calls), which is how a deployed day-ahead forecaster runs. Each call includes copying the result back to the CPU, so GPU work is finished before the timer stops.

In [ ]:
def inference_times(fn, repeats=3, single_calls=200):
    batch, one = X_val[:1024], X_val[:1]
    fn(batch); fn(one)  # warm-up
    start = time.perf_counter()
    for _ in range(repeats):
        fn(batch)
    batched = (time.perf_counter() - start) * 1e6 / (repeats * len(batch))  # ms per 1000 windows
    singles = []
    for _ in range(single_calls):
        start = time.perf_counter(); fn(one); singles.append(time.perf_counter() - start)
    return {'ms per 1000 windows': batched, 'ms per single forecast': 1000 * np.median(singles)}

timing = pd.DataFrame({name: inference_times(fn) for name, fn in predict_fn.items()}).T
print(f'Inference device: {device}')
timing.round(3)

### Validation comparison

In [ ]:
summary = pd.DataFrame({name: val_score(p) for name, p in val_pred.items()}).T
summary['parameters'] = reference['parameters'].reindex(summary.index)
summary.loc[list(FAMILIES), 'parameters'] = [count_params(tuned[f]) for f in FAMILIES]
summary['train time (s)'] = pd.Series(train_secs)
summary = summary.join(seed_table).join(timing).sort_values('val MAE')
print('Table - Validation scores (kW, measured hours; val MAE / RMSE use midnight origins)')
display(summary.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['C1' if name in FAMILIES else 'C7' for name in summary.index]
ax.barh(summary.index[::-1], summary['val MAE'][::-1], color=colors[::-1])
s2 = summary.loc[[n for n in summary.index[::-1] if n in FAMILIES]]
ax.errorbar(s2['val MAE mean (3 seeds)'], [list(summary.index[::-1]).index(n) for n in s2.index],
            xerr=s2['val MAE std (3 seeds)'], fmt='o', color='k', ms=3, capsize=3, label='mean +/- std over 3 seeds')
ax.legend(fontsize=8, loc='lower right')
ax.set(xlabel='Validation MAE (kW, midnight origins, measured hours)',
       title='Fig. 3 - Stage 2 models (orange, seed 42) vs Stage 1 references (grey)')
ax.set_xlim(summary['val MAE'].min() * 0.95, summary['val MAE'].max() * 1.02)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

### Error by forecast hour

In [ ]:
top = [f for f in summary.index if f in FAMILIES][:3]
fig, ax = plt.subplots(figsize=(12, 4))
for name in top + ['Ridge', stage1_name, 'Seasonal naive (week)']:
    err = np.where(DAILY, np.abs(val_pred[name] - Y_kw['val']), np.nan)[daily_origin['val']]
    ax.plot(np.arange(1, HORIZON + 1), np.nanmean(err, axis=0), marker='.', label=name,
            ls='-' if name in FAMILIES else '--')
ax.set(xlabel='Hours ahead', ylabel='MAE (kW)', title='Fig. 4 - Validation error by forecast hour (midnight origins, measured hours)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Forecast vs actual
Three midnight-origin days spread across the validation period.

In [ ]:
days = np.flatnonzero(daily_origin['val'])
picks = days[[len(days) // 4, len(days) // 2, 3 * len(days) // 4]]
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, i in zip(axes, picks):
    hours = features.index[t_val[i]] + pd.to_timedelta(np.arange(HORIZON), 'h')
    ax.plot(hours, Y_kw['val'][i], 'k', lw=2.5, label='actual')
    for name in top[:1] + ['Ridge', stage1_name]:
        ax.plot(hours, val_pred[name][i], lw=1.2, label=name)
    ax.set_title(f'{hours[0]:%a %d %b %Y}'); ax.tick_params(axis='x', rotation=45); ax.grid(alpha=0.3)
axes[0].set_ylabel('Global_active_power (kW)')
axes[0].legend(fontsize=8)
fig.suptitle('Fig. 5 - Day-ahead forecasts vs actual load (validation)')
plt.tight_layout(); plt.show()

### Cost: accuracy vs size, training time and inference time

In [ ]:
s2 = summary.loc[list(FAMILIES)]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, xlabel in [(axes[0], 'parameters', 'Parameters (log)'), (axes[1], 'train time (s)', 'Training time of the selected config (s, log)'),
                        (axes[2], 'ms per single forecast', 'Single-forecast latency (ms, log)')]:
    ax.scatter(s2[col], s2['val MAE'], color='C1')
    for name, r in s2.iterrows():
        ax.annotate(name, (r[col], r['val MAE']), fontsize=7, xytext=(4, 3), textcoords='offset points')
    for ref, ls in [('Ridge', '--'), (stage1_name, ':')]:
        ax.axhline(summary.loc[ref, 'val MAE'], color='grey', ls=ls, lw=1, label=ref)
    ax.set(xscale='log', xlabel=xlabel, ylabel='val MAE (kW)'); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
fig.suptitle('Fig. 6 - Accuracy against model size, training cost and inference latency (lower-left is better)')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
order = timing.sort_values('ms per single forecast').index
for ax, col in zip(axes, timing.columns):
    ax.barh(order[::-1], timing.loc[order[::-1], col], color=['C1' if n in FAMILIES else 'C7' for n in order[::-1]])
    ax.set(xlabel=col, xscale='log'); ax.grid(alpha=0.3, axis='x')
fig.suptitle(f'Fig. 7 - Inference time on {device}')
plt.tight_layout(); plt.show()

## Notes
* Selection uses validation only: the test split is not scored in this notebook.
* `models/stage2_lab6_trials.csv` is the first search (two-stage grid, MSE loss) and supplies the starting configurations; `models/stage2_lab6_refine.csv` is this notebook's search. Both have every trial's configuration, scores, epochs and training time.
* Training times come from the seed-42 retrain of each selected configuration on this machine. Inference times depend on the device, so compare models measured in the same run.